# 🥗 NutriAdapt Model Inference & Verification (`test_inference.ipynb`)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eNVy047/Capstone-PersonalAI-Health-System/blob/main/testing/test_inference.ipynb)

Direct interactive Jupyter / Colab conversion of `test_inference.py`.
- **Hugging Face Model Adapter**: [`narayah90/nutriadapt-qwen-3b-nutrition`](https://huggingface.co/narayah90/nutriadapt-qwen-3b-nutrition)
- **Base LLM**: `Qwen/Qwen2.5-3B-Instruct`
- **Methodology**: 4-bit QLoRA (NF4) with chat template formatting
- **Tasks**:
  1. `explain`: Goal-Grounded Recommendation & Justification
  2. `extract`: Nutrition Breakdown Extraction
  3. `compare`: Two-Meal Dietary Comparison
  4. `custom`: Interactive Custom Meal Testing


In [ ]:
# 1. Install dependencies
!pip install -q -U transformers peft bitsandbytes accelerate jsonschema torch huggingface_hub


In [ ]:
# 2. Setup System Prompt, Goal Rules, Thresholds, and Schemas
import json
import os
import re
import sys

SYSTEM_PROMPT = (
    "You are NutriAdapt, an expert dietary reasoning assistant. Provide factual, evidence-grounded "
    "nutrition advice adhering strictly to the nutrient data given in the prompt. "
    "Never fabricate nutrient values. Return valid JSON when requested."
)

GOAL_RULES = {
    "weight_loss": "low-calorie (<=500 kcal) and high-fiber (>=4g)",
    "heart_healthy": "low-sodium (<500mg)",
    "muscle_gain": "high-protein (>30g)",
}
GOAL_THRESHOLDS = {"500", "4", "30"}

SCHEMAS = {
    "explain": {
        "type": "object",
        "required": ["recommendation", "target_goal", "nutritional_justification", "criteria_met", "factual_compliance"],
        "properties": {
            "recommendation": {"type": "string"},
            "target_goal": {"type": "string"},
            "nutritional_justification": {"type": "string"},
            "criteria_met": {"type": "array", "items": {"type": "string"}},
            "factual_compliance": {"type": "boolean"},
        },
    },
    "extract": {
        "type": "object",
        "required": ["meal", "calories", "macronutrients", "micronutrients"],
        "properties": {
            "meal": {"type": "string"},
            "calories": {"type": "number"},
            "macronutrients": {
                "type": "object",
                "required": ["protein_g", "fat_g", "carbs_g"],
            },
            "micronutrients": {
                "type": "object",
                "required": ["sodium_mg", "fiber_g"],
            },
        },
    },
    "compare": {
        "type": "object",
        "required": ["meal_a", "meal_b", "lower_calorie", "higher_protein", "higher_fiber", "lower_sodium"],
    }
}

print("✓ Prompts, goal rules, and validation schemas initialized.")


In [ ]:
# 3. Helper Functions: JSON Extraction & Numeric Token Grounding
def extract_json(raw_text: str):
    """Strip markdown code block fences and extract JSON object."""
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_text.strip())
    match = re.search(r"(\{.*\})", cleaned, re.DOTALL)
    if match:
        cleaned = match.group(1)
    return json.loads(cleaned)

def extract_numbers(text: str):
    """Extract set of numeric tokens from text for grounding check."""
    return {n.rstrip("0").rstrip(".") if "." in n else n for n in re.findall(r"\d+(?:\.\d+)?", str(text))}

print("✓ Helper functions defined.")


In [ ]:
# 4. Load Model from Hugging Face Hub (Base + LoRA Adapter)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_REPO_ID = "narayah90/nutriadapt-qwen-3b-nutrition"
FORCE_CPU = False

device = "cpu" if FORCE_CPU or not torch.cuda.is_available() else "cuda"
print(f"Device mode: {device.upper()}")
print(f"Loading Tokenizer from: {ADAPTER_REPO_ID}")

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading Base Model:    {BASE_MODEL_ID}")
if device == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        device_map="cpu",
        torch_dtype=torch.float32,
        trust_remote_code=True
    )

print(f"Attaching LoRA Adapter: {ADAPTER_REPO_ID}")
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO_ID)
model.eval()
print("✓ Model and LoRA Adapter successfully initialized!")


In [ ]:
# 5. Core Inference Function with Output Validation & Audit
from jsonschema import validate, ValidationError

def run_inference(model, tokenizer, user_prompt: str, schema_type: str = None, max_new_tokens: int = 256):
    """Run generation and check structured output."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    parsed_json = None
    validation_status = "Skipped"
    try:
        parsed_json = extract_json(response)
        if schema_type and schema_type in SCHEMAS:
            validate(parsed_json, SCHEMAS[schema_type])
            validation_status = "✓ PASSED (Valid schema)"
        else:
            validation_status = "✓ PASSED (Valid JSON)"
    except Exception as e:
        validation_status = f"✗ FAILED ({str(e)})"

    grounding_status = "N/A"
    if parsed_json and schema_type == "explain":
        input_nums = extract_numbers(user_prompt) | GOAL_THRESHOLDS
        justification_text = str(parsed_json.get("nutritional_justification", "")) + " " + " ".join(str(c) for c in parsed_json.get("criteria_met", []))
        claimed_nums = extract_numbers(justification_text)
        invented = claimed_nums - input_nums
        if not invented:
            grounding_status = "✓ ZERO HALLUCINATIONS (All numbers grounded in input)"
        else:
            grounding_status = f"⚠️ Unverified numbers found: {sorted(invented)}"

    return {
        "raw_response": response,
        "parsed_json": parsed_json,
        "validation_status": validation_status,
        "grounding_status": grounding_status
    }

print("✓ run_inference engine ready.")


## 🧪 Test 1: Goal-Grounded Recommendation (`task="explain"`)
Tests why a given meal fits a specific goal (`muscle_gain`, `weight_loss`, or `heart_healthy`) with strict factual compliance.


In [ ]:
# Test 1: Goal-Grounded Explanation
meal_name = "Grilled Salmon Rice Bowl"
goal = "muscle_gain"
calories = 450
protein = 38
fiber = 6
sodium = 420

prompt_explain = (
    f"Explain why '{meal_name}' is recommended for a user with the goal '{goal}'.\n"
    f"Nutritional Facts: {calories:.0f} kcal, {protein:.0f}g protein, {fiber:.0f}g fiber, {sodium:.0f}mg sodium."
)

print("=" * 60)
print("INPUT PROMPT:")
print("=" * 60)
print(prompt_explain)

print("\n" + "=" * 60)
print("RUNNING INFERENCE...")
print("=" * 60)
result_explain = run_inference(model, tokenizer, prompt_explain, schema_type="explain")

print("\nRAW OUTPUT:")
print(result_explain["raw_response"])

print("\n" + "=" * 60)
print("VALIDATION & AUDIT:")
print("=" * 60)
print(f"Schema Validation : {result_explain['validation_status']}")
print(f"Numeric Grounding : {result_explain['grounding_status']}")
if result_explain["parsed_json"]:
    print("\nPARSED JSON:")
    print(json.dumps(result_explain["parsed_json"], indent=2))


## 🧪 Test 2: Nutrition Extraction (`task="extract"`)
Extracts standardized nutrient breakdown from a meal name and ingredients string.


In [ ]:
# Test 2: Nutrition Extraction
meal_extract = "Grilled Salmon Rice Bowl"
ingredients_extract = "salmon^brown rice^broccoli^olive oil"

prompt_extract = (
    f"Extract the standardized nutritional breakdown for the following meal:\n"
    f"Meal: {meal_extract}\n"
    f"Ingredients: {ingredients_extract}"
)

print("=" * 60)
print("INPUT PROMPT:")
print("=" * 60)
print(prompt_extract)

print("\n" + "=" * 60)
print("RUNNING INFERENCE...")
print("=" * 60)
result_extract = run_inference(model, tokenizer, prompt_extract, schema_type="extract")

print("\nRAW OUTPUT:")
print(result_extract["raw_response"])

print("\n" + "=" * 60)
print("VALIDATION & AUDIT:")
print("=" * 60)
print(f"Schema Validation : {result_extract['validation_status']}")
if result_extract["parsed_json"]:
    print("\nPARSED JSON:")
    print(json.dumps(result_extract["parsed_json"], indent=2))


## 🧪 Test 3: Two-Meal Comparison (`task="compare"`)
Compares two candidate meals to determine lower calories, higher protein, higher fiber, and lower sodium.


In [ ]:
# Test 3: Two-Meal Comparison
prompt_compare = (
    "Compare the following two meals.\n"
    "Meal A: Grilled Salmon Bowl (450 kcal, 38g protein, 6g fiber, 420mg sodium)\n"
    "Meal B: Mediterranean Salad (320 kcal, 12g protein, 7g fiber, 480mg sodium)"
)

print("=" * 60)
print("INPUT PROMPT:")
print("=" * 60)
print(prompt_compare)

print("\n" + "=" * 60)
print("RUNNING INFERENCE...")
print("=" * 60)
result_compare = run_inference(model, tokenizer, prompt_compare, schema_type="compare")

print("\nRAW OUTPUT:")
print(result_compare["raw_response"])

print("\n" + "=" * 60)
print("VALIDATION & AUDIT:")
print("=" * 60)
print(f"Schema Validation : {result_compare['validation_status']}")
if result_compare["parsed_json"]:
    print("\nPARSED JSON:")
    print(json.dumps(result_compare["parsed_json"], indent=2))


## 🕹️ Test 4: Custom Input Playground
Put your own custom meal, nutritional numbers, and target goal below to test model outputs!


In [ ]:
# Custom Input Playground - Modify any of these parameters
CUSTOM_MEAL = "Paneer Tikka Salad"
CUSTOM_GOAL = "muscle_gain"  # Options: "muscle_gain", "weight_loss", "heart_healthy"
CUSTOM_CALORIES = 420.0
CUSTOM_PROTEIN = 36.0
CUSTOM_FIBER = 5.0
CUSTOM_SODIUM = 390.0

custom_prompt = (
    f"Explain why '{CUSTOM_MEAL}' is recommended for a user with the goal '{CUSTOM_GOAL}'.\n"
    f"Nutritional Facts: {CUSTOM_CALORIES:.0f} kcal, {CUSTOM_PROTEIN:.0f}g protein, {CUSTOM_FIBER:.0f}g fiber, {CUSTOM_SODIUM:.0f}mg sodium."
)

print("=" * 60)
print("CUSTOM INPUT PROMPT:")
print("=" * 60)
print(custom_prompt)

print("\n" + "=" * 60)
print("RUNNING INFERENCE...")
print("=" * 60)
result_custom = run_inference(model, tokenizer, custom_prompt, schema_type="explain")

print("\nRAW OUTPUT:")
print(result_custom["raw_response"])

print("\n" + "=" * 60)
print("VALIDATION & AUDIT:")
print("=" * 60)
print(f"Schema Validation : {result_custom['validation_status']}")
print(f"Numeric Grounding : {result_custom['grounding_status']}")
if result_custom["parsed_json"]:
    print("\nPARSED JSON:")
    print(json.dumps(result_custom["parsed_json"], indent=2))
